In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

from ml.data import load_traffic, load_schedule, load_points
from ml.predict import DelayPredictor
from ml.data import load_traffic, load_schedule, load_points
from ml.predict import DelayPredictor
import time
import csv

In [12]:
DIR            = Path(".")
TRAFFIC_FILE   = DIR / "traffic_valid.csv"
SCHEDULE_FILE  = DIR / "schedule_plan_valid.csv"
POINTS_FILE    = DIR / "points.csv"
MODEL_DIR      = DIR / "artifacts" / "model"

In [13]:
raw = pd.read_csv(TRAFFIC_FILE)
raw["tr_id"] = pd.to_numeric(raw["tr_id"], errors="coerce")
raw = raw.dropna(subset=["tr_id"])
raw["tr_id"] = raw["tr_id"].astype("int64")

TRAFFIC_CLEAN = DIR / "traffic_valid_clean.csv"
raw.to_csv(TRAFFIC_CLEAN, index=False)

In [5]:
traffic_df  = load_traffic(TRAFFIC_FILE)
schedule_df = load_schedule(SCHEDULE_FILE)
points_df   = load_points(POINTS_FILE)

print("traffic: ", traffic_df.shape)
print("schedule:", schedule_df.shape)
print("points:  ", points_df.shape)

traffic:  (86532, 7)
schedule: (5558, 8)
points:   (151, 8)


In [14]:
pr = DelayPredictor(MODEL_DIR)

t0 = time.time()
pr.set_context(traffic_df, schedule_df)
print(f"Контекст: {len(pr.ctx)} ТС, {time.time()-t0:.1f} c")

pred = pr.predict(points_df)
print(f"pred: shape {pred.shape}, mean {pred.mean():.1f}, std {pred.std():.1f}")
print(f"min: {pred.min():.1f}, max: {pred.max():.1f}")

Контекст: 13 ТС, 3.3 c
pred: shape (151,), mean 55.7, std 110.9
min: -162.7, max: 427.3


# Предсказание для конкретного автобуса

In [ ]:
i = 0
row = points_df.iloc[[i]]

p_one = pr.predict(row)[0]

print("sample_id:      ", row.sample_id.iat[0])
print("tr_id:          ", row.tr_id.iat[0])
print("T:              ", row["T"].iat[0])
print("target_stop_id: ", row.target_stop_id.iat[0])
print("target_time:    ", row.target_time_begin.iat[0])
print("cur_dev_s:      ", row.cur_dev_s.iat[0])
print("prediction (s): ", round(float(p_one), 1))

sample_id:       131672_1767670500
tr_id:           131672
T:               2026-01-06 03:35:00
target_stop_id:  53700172828
target_time:     2026-01-06 03:50:00
cur_dev_s:       274.0
prediction (s):  -13.2
